# 승인된 행신 가격 자료 DB 테스트
사용자가 제공한 행신 SQL 자료만 별도 테스트 DB에서 읽습니다. 다른 실제 데이터 사용 승인은 아닙니다.
매장 8개/옵션 57개. Gemini 호출 없음. 검증일·좌표·판매 상태가 없어 현재 판매 추천으로 확정할 수 없습니다.
기본 스위치는 False이며, 실행하려면 아래 RUN_HAENGSIN_TEST=True로 변경하세요.
DB: coffee_station_haengsin_test. 모의 개발 노트북은 별도로 유지합니다.


In [ ]:
from pathlib import Path
import sys
cwd = Path.cwd().resolve()
AI_ROOT = next(p for base in (cwd, *cwd.parents) for p in (base, base / 'ai-jobs')
               if (p / 'app/review_summary.py').is_file())
if str(AI_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_ROOT))


In [ ]:
RUN_HAENGSIN_TEST = False
if not RUN_HAENGSIN_TEST:
    print('행신 DB 조회 대기: 실행 시 True로 변경하세요.')
else:
    from contextlib import closing
    from app.repositories.haengsin import connect, search_prices, cheapest_by_store
    with closing(connect()) as db, closing(db.cursor()) as cursor:
        cursor.execute('SELECT COUNT(*) AS n FROM import_haengsin_stores')
        assert cursor.fetchone()['n'] == 8
        cursor.execute('SELECT COUNT(*) AS n FROM import_haengsin_prices')
        assert cursor.fetchone()['n'] == 57
    for menu in ['아메리카노', '카페라떼']:
        rows = search_prices(menu_name=menu, max_price=4000, temperature='ICE')
        assert rows and all(r['menu_name']==menu and r['price_won']<=4000 and r['temperature']=='ICE' for r in rows)
        assert all(r['price_basis'] != 'unverified' and r['source_url'] for r in rows)
        print(menu, '4000원 이하 ICE 옵션:', len(rows))
        for row in cheapest_by_store(rows):
            print(row['store_code'], row['store_name'], row['product_name'], row['size_label'],
                  row['price_won'], row['price_basis'])
        exact = search_prices(menu_name=menu, max_price=rows[0]['price_won'], temperature='ICE')
        assert any(r['price_won']==rows[0]['price_won'] for r in exact)
        assert not search_prices(menu_name=menu, max_price=0)
    all_rows = search_prices(menu_name='아메리카노', include_unverified=True) + search_prices(menu_name='카페라떼', include_unverified=True)
    assert len(all_rows)==57
    assert sum(r['price_basis']=='unverified' for r in all_rows)==2
    assert all(r['latitude'] is None and r['longitude'] is None for r in all_rows)
    print('PASS: 행신 DB 검색/경계값/빈 결과/출처/좌표 누락 확인. 현재 판매 여부는 미확인입니다.')
